# S1 Monthly Universe Reshuffle (H-015)

Self-contained build + train for the **monthly ADV reshuffle** arm. Does **not** overwrite
baseline panels under `01_data/data_files/s1_equities/` or production predictions.

**Universe rule (buffered 100/120):** on the first week-start of each calendar month,
rank feature-complete PIT S&P 500 names by trailing 20-day ADV; enter at rank ≤ 100;
keep incumbents while rank ≤ 120; hold until the next reshuffle. Retrain slim-ffill Ridge
on the monthly cross-section. IS dates match `s1_factor_panel_train.parquet`.

**Primary metric (notebook 09):** OOS Sharpe after the shared IS end — do not retune STAR params here.

All expensive stages write `s1_reshuffle_*` caches and short-circuit on cache hit unless `FORCE_REBUILD*` is set.

Set ``ADD_GDELT_FEATURES=False`` in config to skip BigQuery GDELT fetch and drop ``gdelt_*`` from the model feature set.


## 0. Imports & Config


In [10]:
from __future__ import annotations

import os
import sys
import shutil
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from data.ingestion.alternative_data.fama_french_fetcher import fetch_ff_factors_daily
from data.ingestion.alternative_data.sentiment.gdelt_fetcher import (
    DEFAULT_COMPANY_NAME_MAP,
    build_gdelt_alias_table,
    fetch_gdelt_sentiment_daily,
)
from data.ingestion.equity_fetcher import DEFAULT_CACHE_DIR, _download_ohlcv, fetch_ohlcv
from data.ingestion.sp500_universe import load_sp500_snapshots
from data.processing.cleaner import forward_fill_panel
from data.processing.feature_implementation.beta_features import market_return_frame
from data.processing.feature_implementation.momentum import add_raw_momentum
from data.processing.s1_feature_store import (
    add_beta_factors,
    add_gdelt_sentiment_factors,
    add_gross_profitability_factors,
    add_short_flow_factors,
    add_size_value_factors,
    add_volume_factors,
    drop_beta_workspace,
)
from models.s1_equities.training_common import (
    LABEL_COL,
    TARGET_COL,
    add_cs_pct_target,
    attach_scores,
    chronological_is_split,
    cs_rank_features,
    default_paths,
    drop_nonfinite_labels,
    ic_segment_table,
    mean_date_ic,
    week_start_dates,
)

NOTEBOOK_DIR = os.path.join(
    ROOT, "03_models", "s1_equities", "model_tests", "extras"
)
ART_DIR = os.path.join(NOTEBOOK_DIR, "artifacts")
os.makedirs(ART_DIR, exist_ok=True)

BASELINE_TRAIN_PANEL = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)
PATHS = default_paths(ROOT)

OHLCV_CACHE = os.path.join(ROOT, "01_data", "cache", "s1_reshuffle_sp500_ohlcv.parquet")
MEMBERSHIP_PATH = os.path.join(ART_DIR, "s1_reshuffle_membership.parquet")
PANEL_FULL_PATH = os.path.join(ART_DIR, "s1_reshuffle_panel_full.parquet")
PANEL_TRAIN_PATH = os.path.join(ART_DIR, "s1_reshuffle_panel_train.parquet")
SV_CACHE = os.path.join(ROOT, "01_data", "cache", "s1_reshuffle_sv_sec.parquet")
GP_CACHE = os.path.join(ROOT, "01_data", "cache", "s1_reshuffle_gp_sec.parquet")
SHORT_CACHE = os.path.join(ROOT, "01_data", "cache", "s1_reshuffle_short_flow.parquet")
FILING_CACHE = os.path.join(ROOT, "01_data", "cache", "s1_reshuffle_filing_clock.parquet")
GDELT_DAILY_CACHE = os.path.join(ROOT, "01_data", "cache", "s1_reshuffle_gdelt_daily.parquet")
GDELT_OVERLAY_MAP = os.path.join(ART_DIR, "gdelt_company_name_map_reshuffle.csv")
ENG_FULL_PATH = os.path.join(ART_DIR, "s1_reshuffle_engineered_features_full.parquet")
ENG_TRAIN_PATH = os.path.join(ART_DIR, "s1_reshuffle_engineered_features_train.parquet")
PRED_PATH = os.path.join(PATHS["model_dir"], "s1_reshuffle_ridge_predictions.parquet")
os.makedirs(PATHS["model_dir"], exist_ok=True)

USE_SHIPPED_GDELT_MAP = False
OVERLAY_MAP_PATH = (
    DEFAULT_COMPANY_NAME_MAP if USE_SHIPPED_GDELT_MAP else GDELT_OVERLAY_MAP
)

FORCE_REBUILD = False
FORCE_REBUILD_OHLCV = False
FORCE_REBUILD_PANEL = False
FORCE_REBUILD_ALT = False
FORCE_REBUILD_GDELT = False
# Set False to skip BigQuery GDELT fetch + gdelt_* model features (quota / cost).
ADD_GDELT_FEATURES = False
FORCE_REBUILD_FEATURES = False
FORCE_REBUILD_MEMBERSHIP = False

OHLCV_START = "2013-01-01"
PANEL_START = "2015-01-01"
UNIVERSE_N = 100
EXIT_RANK = 120
LOOKBACK_DAYS = 20
PREFILTER_N = 250
MIN_RANKING_BARS = 10
BENCHMARK = "RSP"
UNIVARIATE_BENCHMARK = BENCHMARK.lower()
FFILL_LIMIT = 5
MIN_ARTICLES_OK = 50

FEATURE_COLS = [
    "raw_momentum_252_5",
    "smart_residual_mom_189_42",
    "rel_downside_beta_252",
    "rel_upside_beta_63",
    "smart_beta_hml_252",
    "downside_beta_42",
    "upside_beta_42",
    "size_mom_126",
    "val_roc_pb_252",
    "val_roc_pe_252",
    "log_mcap",
    "val_mom_dist_252_21",
    "gross_profitability",
    "filing_clock_expected_until",
    "short_flow_ratio",
    "market_corr",
    "beta_mkt_interact",
    "abnormal_volume",
]
GDELT_FEATURE_COLS = [
    "gdelt_tone_x_attention_21",
    "gdelt_attention_5",
]
if ADD_GDELT_FEATURES:
    FEATURE_COLS = FEATURE_COLS + GDELT_FEATURE_COLS

VAL_FRAC = 0.15
EMBARGO_WEEKS = 1
ALPHA_GRID = [0.1, 1.0, 10.0, 100.0]
RANDOM_SEED = 42

KEY_COLS = ["date", "ticker", "feature_date"]
CONTEXT_COLS = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "fwd_ret_1",
    "fwd_ret_5",
    "fwd_ret_21",
]


def _cache_ok(path: str, force: bool) -> bool:
    return (not FORCE_REBUILD) and (not force) and os.path.isfile(path)


def _ffill_by_ticker(frame: pd.DataFrame, cols: list[str], limit: int) -> pd.DataFrame:
    out = frame.copy()
    present = [c for c in cols if c in out.columns]
    if not present:
        return out
    out = out.sort_values(["ticker", "date"])
    out[present] = out.groupby("ticker", sort=False)[present].ffill(limit=limit)
    return out


def _rename_stem(frame: pd.DataFrame, stem: str, target: str) -> pd.DataFrame:
    if target in frame.columns:
        if stem in frame.columns and stem != target:
            return frame.drop(columns=[stem])
        return frame
    if stem not in frame.columns:
        raise ValueError(
            f"expected store column {stem!r} to rename to {target!r}; neither present"
        )
    return frame.rename(columns={stem: target})


print(f"ROOT={ROOT}")
print(f"ART_DIR={ART_DIR}")
print(f"FORCE_REBUILD={FORCE_REBUILD}  OVERLAY_MAP_PATH={OVERLAY_MAP_PATH}")
print(f"ADD_GDELT_FEATURES={ADD_GDELT_FEATURES}")
print(f"FEATURE_COLS={len(FEATURE_COLS)}")


ROOT=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio
ART_DIR=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_tests\extras\artifacts
FORCE_REBUILD=False  OVERLAY_MAP_PATH=c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_tests\extras\artifacts\gdelt_company_name_map_reshuffle.csv
ADD_GDELT_FEATURES=False
FEATURE_COLS=18


## 1. Superset OHLCV (PIT S&P union)


In [11]:
baseline_is_dates = pd.Index(
    pd.to_datetime(pd.read_parquet(BASELINE_TRAIN_PANEL, columns=["date"])["date"])
    .dropna()
    .unique()
).sort_values()
print(
    f"Baseline IS dates: {baseline_is_dates.min().date()} -> "
    f"{baseline_is_dates.max().date()} ({len(baseline_is_dates):,})"
)

if _cache_ok(OHLCV_CACHE, FORCE_REBUILD_OHLCV):
    print(f"CACHE HIT: {OHLCV_CACHE}")
    raw_ohlcv = pd.read_parquet(OHLCV_CACHE)
else:
    print(f"CACHE MISS: building PIT S&P union OHLCV -> {OHLCV_CACHE}")
    snaps = load_sp500_snapshots()
    snaps = snaps.loc[pd.to_datetime(snaps["date"]) >= pd.Timestamp("2012-01-01")]
    tickers: set[str] = set()
    for tstr in tqdm(snaps["tickers"], desc="PIT union", file=sys.stderr):
        tickers.update(x.strip().upper() for x in str(tstr).split(",") if x.strip())
    ticker_list = sorted(tickers)
    print(f"Unique PIT members since 2012: {len(ticker_list)}")
    end_ts = pd.Timestamp.now().normalize()
    start_ts = pd.Timestamp(OHLCV_START)
    raw_ohlcv = _download_ohlcv(
        ticker_list,
        start_ts,
        end_ts,
        cache_dir=DEFAULT_CACHE_DIR,
        cache_label="s1_reshuffle_sp500_union",
    )
    raw_ohlcv["date"] = pd.to_datetime(raw_ohlcv["date"])
    raw_ohlcv["ticker"] = raw_ohlcv["ticker"].astype(str).str.strip().str.upper()
    os.makedirs(os.path.dirname(OHLCV_CACHE), exist_ok=True)
    raw_ohlcv.to_parquet(OHLCV_CACHE, index=False)
    print(
        f"Wrote {OHLCV_CACHE}  rows={len(raw_ohlcv):,}  "
        f"tickers={raw_ohlcv['ticker'].nunique()}"
    )

raw_ohlcv["date"] = pd.to_datetime(raw_ohlcv["date"])
raw_ohlcv["ticker"] = raw_ohlcv["ticker"].astype(str).str.strip().str.upper()
print(
    f"OHLCV: rows={len(raw_ohlcv):,} tickers={raw_ohlcv['ticker'].nunique()} "
    f"span={raw_ohlcv['date'].min().date()} -> {raw_ohlcv['date'].max().date()}"
)


Baseline IS dates: 2015-01-05 -> 2023-02-03 (2,036)
CACHE HIT: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\cache\s1_reshuffle_sp500_ohlcv.parquet
OHLCV: rows=2,025,551 tickers=649 span=2013-01-02 -> 2026-08-04


## 2. Trade-date panel (dual save)

Same contract as `s1_factor_panel.ipynb`: `date` / `feature_date` / lagged hlcv / `fwd_ret_*`.
Train cut uses baseline `s1_factor_panel_train.parquet` dates (not a new fraction).
Prefilter: union of monthly top-`PREFILTER_N` by ADV before engineering.


In [12]:
if _cache_ok(PANEL_FULL_PATH, FORCE_REBUILD_PANEL) and _cache_ok(
    PANEL_TRAIN_PATH, FORCE_REBUILD_PANEL
):
    print(f"CACHE HIT: {PANEL_FULL_PATH}")
    panel = pd.read_parquet(PANEL_FULL_PATH)
    panel["date"] = pd.to_datetime(panel["date"])
    panel["feature_date"] = pd.to_datetime(panel["feature_date"])
else:
    print("CACHE MISS: building trade-date panel")
    work = raw_ohlcv.sort_values(["ticker", "date"], kind="mergesort").copy()
    work["date"] = pd.to_datetime(work["date"])
    work["feature_date"] = work.groupby("ticker", sort=False)["date"].shift(1)
    for col in ("high", "low", "close", "volume"):
        work[col] = work.groupby("ticker", sort=False)[col].shift(1)
    work = work.dropna(subset=["feature_date"]).reset_index(drop=True)
    for h in (1, 5, 21):
        work[f"fwd_ret_{h}"] = work.groupby("ticker", sort=False)["open"].transform(
            lambda s, hh=h: s.shift(-hh) / s - 1.0
        )
    panel = work.loc[work["date"] >= pd.Timestamp(PANEL_START)].copy()
    required_cols = {
        "date",
        "ticker",
        "feature_date",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "fwd_ret_1",
        "fwd_ret_5",
        "fwd_ret_21",
    }
    missing = required_cols - set(panel.columns)
    if missing:
        raise ValueError(f"trade-date panel missing columns: {sorted(missing)}")
    if not panel["feature_date"].lt(panel["date"]).all():
        raise ValueError("feature_date must be strictly before date on every row")

    panel = panel.sort_values(["ticker", "date"])
    panel["_dv"] = panel["close"].astype(float) * panel["volume"].astype(float)
    panel["_adv"] = panel.groupby("ticker", sort=False)["_dv"].transform(
        lambda s: s.rolling(LOOKBACK_DAYS, min_periods=MIN_RANKING_BARS).mean()
    )
    week_starts = week_start_dates(panel["date"])
    month_keys = pd.Series(week_starts).dt.to_period("M")
    reshuffle_dates = (
        pd.DataFrame({"date": week_starts, "ym": month_keys})
        .groupby("ym", sort=True)["date"]
        .min()
        .tolist()
    )
    prefilter: set[str] = set()
    for d in tqdm(reshuffle_dates, desc="ADV prefilter", file=sys.stderr):
        snap = panel.loc[panel["date"] == d, ["ticker", "_adv"]].dropna()
        if snap.empty:
            continue
        top = snap.sort_values("_adv", ascending=False).head(PREFILTER_N)["ticker"]
        prefilter.update(top.astype(str).str.upper().tolist())
    print(f"Prefilter union (monthly top-{PREFILTER_N}): {len(prefilter)} tickers")
    panel = panel.loc[panel["ticker"].isin(prefilter)].drop(
        columns=["_dv", "_adv"], errors="ignore"
    ).copy()

    train_panel = panel.loc[panel["date"].isin(baseline_is_dates)].copy()
    if train_panel.empty:
        raise ValueError("reshuffle train panel empty — check date overlap with baseline IS")
    train_max = pd.to_datetime(train_panel["date"]).max()
    base_max = baseline_is_dates.max()
    if train_max != base_max:
        raise ValueError(
            f"reshuffle train max date {train_max.date()} != baseline {base_max.date()}"
        )
    panel.to_parquet(PANEL_FULL_PATH, index=False)
    train_panel.to_parquet(PANEL_TRAIN_PATH, index=False)
    print(f"Wrote {PANEL_FULL_PATH}  rows={len(panel):,}")
    print(f"Wrote {PANEL_TRAIN_PATH}  rows={len(train_panel):,}")

panel["date"] = pd.to_datetime(panel["date"])
panel["feature_date"] = pd.to_datetime(panel["feature_date"])
panel["ticker"] = panel["ticker"].astype(str).str.strip().str.upper()
print(
    f"Panel: rows={len(panel):,} tickers={panel['ticker'].nunique()} "
    f"span={panel['date'].min().date()} -> {panel['date'].max().date()}"
)


CACHE HIT: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_tests\extras\artifacts\s1_reshuffle_panel_full.parquet
Panel: rows=1,528,725 tickers=547 span=2015-01-02 -> 2026-08-04


## 3. GDELT checkpoint (pause between A and C)

Skipped entirely when ``ADD_GDELT_FEATURES`` is False.

Cell A fetches (or loads cache). Cell B prints unmapped / zero-article tickers so you can
edit `gdelt_company_name_map_reshuffle.csv`. Cell C re-fetches **only** still-missing tickers.

**Cost:** editing the alias CSV invalidates GDELT month-chunk cache keys — batch all edits into one visit before Cell C.


### 3.1 Cell A — GDELT pass 1


In [13]:
superset_tickers = sorted(panel["ticker"].unique().tolist())
gdelt_start = max(pd.Timestamp(PANEL_START), pd.Timestamp("2015-02-18"))
gdelt_end = panel["date"].max()

if not ADD_GDELT_FEATURES:
    print("ADD_GDELT_FEATURES=False — skipping GDELT pass 1 / BigQuery")
    gdelt_daily = pd.DataFrame(columns=["date", "ticker", "median_tone", "n_articles"])
else:
    if (not USE_SHIPPED_GDELT_MAP) and (not os.path.isfile(GDELT_OVERLAY_MAP)):
        os.makedirs(os.path.dirname(GDELT_OVERLAY_MAP), exist_ok=True)
        shutil.copyfile(DEFAULT_COMPANY_NAME_MAP, GDELT_OVERLAY_MAP)
        print(f"Seeded overlay from shipped map: {GDELT_OVERLAY_MAP}")

    if _cache_ok(GDELT_DAILY_CACHE, FORCE_REBUILD_GDELT):
        print(f"CACHE HIT: {GDELT_DAILY_CACHE}")
        gdelt_daily = pd.read_parquet(GDELT_DAILY_CACHE)
    else:
        print(f"CACHE MISS: GDELT pass 1 for {len(superset_tickers)} tickers")
        gdelt_daily = fetch_gdelt_sentiment_daily(
            superset_tickers,
            start_date=gdelt_start.date(),
            end_date=gdelt_end.date(),
            cache_dir=DEFAULT_CACHE_DIR,
            company_name_map_path=OVERLAY_MAP_PATH,
            use_bigquery=True,
            live_n_files=0,
            resume=True,
        )
        gdelt_daily.to_parquet(GDELT_DAILY_CACHE, index=False)
        print(f"Wrote {GDELT_DAILY_CACHE}  rows={len(gdelt_daily):,}")

    gdelt_daily["date"] = pd.to_datetime(gdelt_daily["date"])
    gdelt_daily["ticker"] = gdelt_daily["ticker"].astype(str).str.strip().str.upper()
    print(
        f"GDELT daily: rows={len(gdelt_daily):,} tickers={gdelt_daily['ticker'].nunique()} "
        f"span={gdelt_daily['date'].min().date() if len(gdelt_daily) else 'n/a'} -> "
        f"{gdelt_daily['date'].max().date() if len(gdelt_daily) else 'n/a'}"
    )


ADD_GDELT_FEATURES=False — skipping GDELT pass 1 / BigQuery


### 3.2 Cell B — diagnostic pause (edit overlay CSV, then run Cell C)


In [14]:
if not ADD_GDELT_FEATURES:
    print("ADD_GDELT_FEATURES=False — skipping GDELT mapping diagnostic")
    MISS_TICKERS = []
else:
    alias_df = build_gdelt_alias_table(
        superset_tickers,
        cache_dir=DEFAULT_CACHE_DIR,
        company_name_map_path=OVERLAY_MAP_PATH,
    )
    hit_counts = (
        gdelt_daily.groupby("ticker")["n_articles"].sum()
        if not gdelt_daily.empty
        else pd.Series(dtype=float)
    )

    rows = []
    for _, row in alias_df.iterrows():
        t = str(row["ticker"]).upper()
        title = str(row.get("title") or "")
        n_aliases = int(row.get("n_aliases") or 0)
        aliases = str(row.get("aliases") or "")
        n_art = float(hit_counts.get(t, 0) or 0)
        if title == "" or n_aliases == 0:
            kind = "NO_SEC_IDENTITY"
        elif n_art <= 0:
            kind = "ZERO_ARTICLES"
        elif n_art < MIN_ARTICLES_OK:
            kind = "THIN_COVERAGE"
        else:
            continue
        rows.append(
            {
                "kind": kind,
                "ticker": t,
                "title": title,
                "n_aliases": n_aliases,
                "n_articles": n_art,
                "aliases": aliases,
                "suggested_csv_row": f"{title or t},{t}",
            }
        )

    diag = pd.DataFrame(rows)
    if diag.empty:
        print("No GDELT mapping issues flagged — you can skip Cell C.")
    else:
        for kind in ("NO_SEC_IDENTITY", "ZERO_ARTICLES", "THIN_COVERAGE"):
            part = diag.loc[diag["kind"] == kind]
            print(f"\n=== {kind} ({len(part)}) ===")
            if part.empty:
                continue
            display(part[["ticker", "title", "n_articles", "aliases"]].head(80))
        print("\n--- Paste-ready CSV rows (append to overlay) ---")
        print("company_name,ticker")
        for s in diag["suggested_csv_row"].drop_duplicates():
            print(s)
        print(
            f"\nEdit: {OVERLAY_MAP_PATH}\n"
            "Then run Cell C. Batch all edits before Cell C — alias changes "
            "invalidate GDELT month-chunk cache keys."
        )

    MISS_TICKERS = (
        sorted(
            diag.loc[
                diag["kind"].isin(["NO_SEC_IDENTITY", "ZERO_ARTICLES"]), "ticker"
            ].unique()
        )
        if not diag.empty
        else []
    )
    print(f"\nMISS_TICKERS for Cell C top-up: {len(MISS_TICKERS)}")


ADD_GDELT_FEATURES=False — skipping GDELT mapping diagnostic


### 3.3 Cell C — GDELT top-up for still-missing tickers


In [15]:
if not ADD_GDELT_FEATURES:
    print("ADD_GDELT_FEATURES=False — skipping GDELT top-up")
else:
    before_tickers = set(gdelt_daily["ticker"].unique()) if len(gdelt_daily) else set()
    before_cov = len(before_tickers & set(superset_tickers))

    alias_df2 = build_gdelt_alias_table(
        superset_tickers,
        cache_dir=DEFAULT_CACHE_DIR,
        company_name_map_path=OVERLAY_MAP_PATH,
    )
    hit_counts2 = (
        gdelt_daily.groupby("ticker")["n_articles"].sum()
        if not gdelt_daily.empty
        else pd.Series(dtype=float)
    )
    still_missing = []
    for _, row in alias_df2.iterrows():
        t = str(row["ticker"]).upper()
        n_art = float(hit_counts2.get(t, 0) or 0)
        if n_art <= 0:
            still_missing.append(t)

    if not still_missing:
        print("Nothing to top up — all tickers have >0 GDELT articles (or none flagged).")
    else:
        print(
            f"Top-up fetch for {len(still_missing)} tickers "
            "(second BQ pass if aliases changed)"
        )
        gdelt_top = fetch_gdelt_sentiment_daily(
            still_missing,
            start_date=gdelt_start.date(),
            end_date=gdelt_end.date(),
            cache_dir=DEFAULT_CACHE_DIR,
            company_name_map_path=OVERLAY_MAP_PATH,
            use_bigquery=True,
            live_n_files=0,
            resume=True,
        )
        gdelt_top["date"] = pd.to_datetime(gdelt_top["date"])
        gdelt_top["ticker"] = gdelt_top["ticker"].astype(str).str.strip().str.upper()
        merged = pd.concat([gdelt_daily, gdelt_top], ignore_index=True)
        merged = (
            merged.sort_values(["date", "ticker"])
            .drop_duplicates(subset=["date", "ticker"], keep="last")
            .reset_index(drop=True)
        )
        gdelt_daily = merged
        gdelt_daily.to_parquet(GDELT_DAILY_CACHE, index=False)
        print(f"Overwrote {GDELT_DAILY_CACHE}  rows={len(gdelt_daily):,}")

    after_tickers = set(gdelt_daily["ticker"].unique()) if len(gdelt_daily) else set()
    after_cov = len(after_tickers & set(superset_tickers))
    print(
        f"Coverage tickers with any GDELT row: "
        f"{before_cov} -> {after_cov} / {len(superset_tickers)}"
    )


ADD_GDELT_FEATURES=False — skipping GDELT top-up


## 4. Alt-data fetch + feature engineering (surviving columns)

Uses existing `s1_feature_store` dispatchers with the production kwargs. Writes reshuffle-prefixed
alt-data caches and dual engineered parquets (`*_full` / `*_train`).


In [16]:
if _cache_ok(ENG_FULL_PATH, FORCE_REBUILD_FEATURES) and _cache_ok(
    ENG_TRAIN_PATH, FORCE_REBUILD_FEATURES
):
    print(f"CACHE HIT: {ENG_FULL_PATH}")
    eng = pd.read_parquet(ENG_FULL_PATH)
    eng["date"] = pd.to_datetime(eng["date"])
    eng["feature_date"] = pd.to_datetime(eng["feature_date"])
    missing = [c for c in FEATURE_COLS if c not in eng.columns]
    if missing:
        raise ValueError(
            f"Engineered cache missing FEATURE_COLS={missing}. "
            "Set FORCE_REBUILD_FEATURES=True after toggling ADD_GDELT_FEATURES."
        )
else:
    print("CACHE MISS: engineering 20 features on reshuffle panel")
    eng = panel.copy()
    tickers = sorted(eng["ticker"].unique().tolist())
    start = eng["date"].min()
    end = eng["date"].max()
    # SEC / FINRA / filing alt data are fetched inside add_*_factors below.

    if ADD_GDELT_FEATURES:
        gd = gdelt_daily.rename(columns={"date": "feature_date"}).copy()
        gd["feature_date"] = pd.to_datetime(gd["feature_date"])
        eng = eng.merge(
            gd[["feature_date", "ticker", "median_tone", "n_articles"]],
            on=["feature_date", "ticker"],
            how="left",
        )

    market_ohlcv = fetch_ohlcv(
        BENCHMARK, start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")
    )
    market_returns = market_return_frame(market_ohlcv)
    ff_factors = fetch_ff_factors_daily(
        start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")
    )

    def _eng_raw_momentum() -> None:
        global eng
        eng = add_raw_momentum(eng, lookback=252, skip=5, col="raw_momentum")
        eng = _rename_stem(eng, "raw_momentum", "raw_momentum_252_5")

    def _make_beta_step(target: str, subset_id: str, kwargs: dict, need: str):
        def _step() -> None:
            global eng
            call_kwargs = dict(kwargs)
            call_kwargs["feature_subset"] = [subset_id]
            call_kwargs["benchmark"] = UNIVARIATE_BENCHMARK
            if need == "ff":
                eng = add_beta_factors(
                    eng,
                    market_returns=market_returns,
                    ff_factors=ff_factors,
                    **call_kwargs,
                )
            else:
                eng = add_beta_factors(
                    eng, market_returns=market_returns, **call_kwargs
                )
            if target != subset_id:
                eng = _rename_stem(eng, subset_id, target)

        return _step

    beta_specs = [
        ("rel_downside_beta_252", "rel_downside_beta", {"windows": 252}, "mkt"),
        ("rel_upside_beta_63", "rel_upside_beta", {"windows": 63}, "mkt"),
        ("downside_beta_42", "downside_beta", {"windows": 42}, "mkt"),
        ("upside_beta_42", "upside_beta", {"windows": 42}, "mkt"),
        ("smart_beta_hml_252", "smart_beta_hml", {"windows": 252}, "ff"),
        (
            "smart_residual_mom_189_42",
            "smart_residual_mom",
            {"formation_window": 189, "skip": 42},
            "ff",
        ),
        ("market_corr", "market_corr", {"windows": 252}, "mkt"),
        (
            "beta_mkt_interact",
            "beta_mkt_interact",
            {"windows": 252, "mkt_horizon": 5},
            "mkt",
        ),
    ]

    _sv_attached = {"done": False}

    def _make_sv_step(target: str, subset_id: str, kwargs: dict):
        def _step() -> None:
            global eng
            exists = _sv_attached["done"]
            eng = add_size_value_factors(
                eng,
                feature_subset=[subset_id],
                size_value_data_exists=exists,
                **kwargs,
            )
            if not exists and subset_id != "amihud":
                eng = _ffill_by_ticker(
                    eng,
                    [
                        "market_cap",
                        "pe",
                        "pb",
                        "shares_outstanding",
                        "book_equity",
                        "eps_ttm",
                    ],
                    FFILL_LIMIT,
                )
                _sv_attached["done"] = True
            if target != subset_id:
                eng = _rename_stem(eng, subset_id, target)

        return _step

    sv_specs = [
        ("size_mom_126", "size_mom", {"window": 126}),
        ("val_roc_pb_252", "val_roc_pb", {"window": 252}),
        ("val_roc_pe_252", "val_roc_pe", {"window": 252}),
        ("log_mcap", "log_mcap", {}),
        (
            "val_mom_dist_252_21",
            "val_mom_dist",
            {"mom_lookback": 252, "mom_skip": 21},
        ),
    ]

    def _drop_ws() -> None:
        global eng
        eng = drop_beta_workspace(eng)

    def _eng_gp() -> None:
        global eng
        eng = add_gross_profitability_factors(
            eng,
            feature_subset=["gross_profitability"],
            normalize=True,
            gross_profitability_data_exists=False,
        )
        eng = _ffill_by_ticker(
            eng,
            ["gp_asset", "gross_profit_ttm", "assets"],
            FFILL_LIMIT,
        )

    def _eng_filing() -> None:
        global eng
        eng = add_short_flow_factors(
            eng,
            feature_subset=["filing_expected_until"],
            filing_clock_data_exists=False,
        )

    def _eng_short() -> None:
        global eng
        eng = add_short_flow_factors(
            eng,
            feature_subset=["ratio"],
            short_volume_data_exists=False,
        )
        eng = _ffill_by_ticker(
            eng,
            ["short_volume", "short_exempt_volume", "total_volume"],
            FFILL_LIMIT,
        )

    def _eng_abn_vol() -> None:
        global eng
        eng = add_volume_factors(
            eng,
            feature_subset=["abnormal_volume"],
            smooth_window=5,
            baseline_window=60,
        )

    def _eng_gdelt_txa() -> None:
        global eng
        eng = add_gdelt_sentiment_factors(
            eng,
            feature_subset=["tone_x_attention"],
            window=21,
            sentiment_data_exists=True,
        )
        eng = _rename_stem(eng, "gdelt_tone_x_attention", "gdelt_tone_x_attention_21")

    def _eng_gdelt_att() -> None:
        global eng
        eng = add_gdelt_sentiment_factors(
            eng,
            feature_subset=["attention"],
            window=5,
            sentiment_data_exists=True,
        )
        eng = _rename_stem(eng, "gdelt_attention", "gdelt_attention_5")

    steps: list[tuple[str, object]] = [("raw_momentum_252_5", _eng_raw_momentum)]
    for target, subset_id, kwargs, need in beta_specs:
        steps.append((target, _make_beta_step(target, subset_id, kwargs, need)))
    steps.append(("_drop_beta_workspace", _drop_ws))
    for target, subset_id, kwargs in sv_specs:
        steps.append((target, _make_sv_step(target, subset_id, kwargs)))
    steps.extend(
        [
            ("gross_profitability", _eng_gp),
            ("filing_clock_expected_until", _eng_filing),
            ("short_flow_ratio", _eng_short),
            ("abnormal_volume", _eng_abn_vol),
        ]
    )
    if ADD_GDELT_FEATURES:
        steps.extend(
            [
                ("gdelt_tone_x_attention_21", _eng_gdelt_txa),
                ("gdelt_attention_5", _eng_gdelt_att),
            ]
        )

    t0 = time.perf_counter()
    for name, fn in tqdm(steps, desc="Reshuffle factors", unit="factor", file=sys.stderr):
        fn()
    missing = [c for c in FEATURE_COLS if c not in eng.columns]
    if missing:
        raise ValueError(f"FEATURE_COLS missing after engineering: {missing}")

    out = eng.loc[:, KEY_COLS + CONTEXT_COLS + FEATURE_COLS].copy()
    out["date"] = pd.to_datetime(out["date"])
    out["feature_date"] = pd.to_datetime(out["feature_date"])
    train_out = out.loc[out["date"].isin(baseline_is_dates)].copy()
    out.to_parquet(ENG_FULL_PATH, index=False)
    train_out.to_parquet(ENG_TRAIN_PATH, index=False)
    print(
        f"Wrote {ENG_FULL_PATH} rows={len(out):,}  "
        f"{ENG_TRAIN_PATH} rows={len(train_out):,}  "
        f"elapsed={time.perf_counter() - t0:.1f}s"
    )
    eng = out

eng["date"] = pd.to_datetime(eng["date"])
eng["feature_date"] = pd.to_datetime(eng["feature_date"])
eng["ticker"] = eng["ticker"].astype(str).str.strip().str.upper()
print(f"Engineered: {eng.shape}  tickers={eng['ticker'].nunique()}")


CACHE HIT: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_tests\extras\artifacts\s1_reshuffle_engineered_features_full.parquet
Engineered: (1528725, 29)  tickers=547


## 5. Monthly buffered membership (filter-then-rank)

Eligible = PIT S&P member **and** complete on all 20 features after within-ticker ffill.
Enter at ADV rank ≤ 100; keep incumbents while rank ≤ 120.


In [17]:
if _cache_ok(MEMBERSHIP_PATH, FORCE_REBUILD_MEMBERSHIP):
    print(f"CACHE HIT: {MEMBERSHIP_PATH}")
    membership = pd.read_parquet(MEMBERSHIP_PATH)
else:
    print("CACHE MISS: computing buffered monthly membership")
    work = eng.sort_values(["ticker", "date"]).copy()
    work = forward_fill_panel(work, columns=FEATURE_COLS, limit=None)
    work["_dv"] = work["close"].astype(float) * work["volume"].astype(float)
    work["_adv"] = work.groupby("ticker", sort=False)["_dv"].transform(
        lambda s: s.rolling(LOOKBACK_DAYS, min_periods=MIN_RANKING_BARS).mean()
    )
    work["_complete"] = work[FEATURE_COLS].notna().all(axis=1)

    snaps = load_sp500_snapshots()
    snaps = snaps.sort_values("date").reset_index(drop=True)
    snap_dates = pd.to_datetime(snaps["date"])

    def pit_members(as_of: pd.Timestamp) -> set[str]:
        elig = snaps.loc[snap_dates <= as_of]
        if elig.empty:
            return set()
        row = elig.iloc[-1]
        return {t.strip().upper() for t in str(row["tickers"]).split(",") if t.strip()}

    week_starts = week_start_dates(work["date"])
    month_keys = pd.Series(week_starts).dt.to_period("M")
    reshuffle_dates = (
        pd.DataFrame({"date": week_starts, "ym": month_keys})
        .groupby("ym", sort=True)["date"]
        .min()
        .tolist()
    )

    prev: set[str] = set()
    rows = []
    max_entry_raw_rank = 0
    for d in tqdm(reshuffle_dates, desc="Membership", file=sys.stderr):
        d = pd.Timestamp(d)
        day = work.loc[work["date"] == d].copy()
        if day.empty:
            continue
        pit = pit_members(d)
        day = day.loc[day["ticker"].isin(pit) & day["_complete"] & day["_adv"].notna()]
        if day.empty:
            continue
        day = day.sort_values("_adv", ascending=False).reset_index(drop=True)
        day["rank"] = np.arange(1, len(day) + 1)
        rank_map = dict(zip(day["ticker"], day["rank"]))

        kept = {t for t in prev if rank_map.get(t, 10**9) <= EXIT_RANK}
        if len(kept) > UNIVERSE_N:
            kept = set(sorted(kept, key=lambda t: rank_map.get(t, 10**9))[:UNIVERSE_N])
        slots = UNIVERSE_N - len(kept)
        for _, r in day.iterrows():
            if slots <= 0:
                break
            t = str(r["ticker"])
            if t in kept:
                continue
            if int(r["rank"]) <= UNIVERSE_N:
                kept.add(t)
                max_entry_raw_rank = max(max_entry_raw_rank, int(r["rank"]))
                slots -= 1
        prev = kept
        for t in sorted(kept):
            rows.append(
                {
                    "reshuffle_date": d,
                    "ticker": t,
                    "adv_rank": rank_map.get(t, np.nan),
                }
            )

    membership = pd.DataFrame(rows)
    membership.to_parquet(MEMBERSHIP_PATH, index=False)
    print(f"Wrote {MEMBERSHIP_PATH}  rows={len(membership):,}")
    print(f"Max ADV rank among new entrants (within eligible): {max_entry_raw_rank}")
    if max_entry_raw_rank >= PREFILTER_N - 5:
        print(
            f"WARNING: max entry rank {max_entry_raw_rank} near PREFILTER_N={PREFILTER_N} "
            "— raise PREFILTER_N and rebuild."
        )

membership["reshuffle_date"] = pd.to_datetime(membership["reshuffle_date"])
membership["ticker"] = membership["ticker"].astype(str).str.strip().str.upper()
print(
    f"Membership: reshuffles={membership['reshuffle_date'].nunique()} "
    f"unique tickers={membership['ticker'].nunique()} "
    f"mean names/month={membership.groupby('reshuffle_date').size().mean():.1f}"
)
display(membership.groupby("reshuffle_date").size().describe().to_frame("n_names"))


CACHE HIT: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_tests\extras\artifacts\s1_reshuffle_membership.parquet
Membership: reshuffles=122 unique tickers=237 mean names/month=100.0


,n_names
count,122.0
mean,100.0
std,0.0
min,100.0
25%,100.0
50%,100.0
75%,100.0
max,100.0


## 6. Train slim-ffill Ridge (universe-masked)

Membership mask applied **before** CS target / feature ranks. `is_research_is` from
baseline train-panel dates. Predictions scored on the masked week-start panel and written
to `s1_reshuffle_ridge_predictions.parquet`.


In [18]:
week_starts = week_start_dates(eng["date"])
rs = (
    membership[["reshuffle_date", "ticker"]]
    .drop_duplicates()
    .sort_values("reshuffle_date")
)
ws = pd.Series(week_starts, name="date")
rs_dates = pd.DatetimeIndex(rs["reshuffle_date"].unique()).sort_values()
asof = pd.Series(rs_dates, index=rs_dates).reindex(ws, method="ffill")
week_to_rs = dict(zip(ws, asof))

pairs = []
for d, rsd in week_to_rs.items():
    if pd.isna(rsd):
        continue
    tickers = rs.loc[rs["reshuffle_date"] == rsd, "ticker"]
    for t in tickers:
        pairs.append({"date": d, "ticker": t})
member_keys = pd.DataFrame(pairs)
member_keys["date"] = pd.to_datetime(member_keys["date"])

df = eng.merge(member_keys, on=["date", "ticker"], how="inner")
df = df.loc[df["date"].isin(week_starts)].copy()
df["is_research_is"] = df["date"].isin(baseline_is_dates)
print(
    f"Masked week panel: rows={len(df):,} weeks={df['date'].nunique()} "
    f"tickers={df['ticker'].nunique()} "
    f"IS_weeks={df.loc[df['is_research_is'], 'date'].nunique()}"
)

df = forward_fill_panel(df, columns=FEATURE_COLS, limit=None)
df = add_cs_pct_target(df)
n_before = len(df)
df = drop_nonfinite_labels(df, [LABEL_COL, TARGET_COL])
print(f"After finite labels: {len(df):,} (from {n_before:,})")

X_ranked = cs_rank_features(df, FEATURE_COLS, fill_value=None)
nan_any = X_ranked.isna().any(axis=1)
n_dropped = int(nan_any.sum())
df = df.loc[~nan_any].copy()
X = X_ranked.loc[~nan_any].copy()
print(f"Complete-case drop: dropped={n_dropped:,}  remain={len(df):,}")

split = chronological_is_split(df, val_frac=VAL_FRAC, embargo_weeks=EMBARGO_WEEKS)
train_dates, val_dates = split.train_dates, split.val_dates
is_end = split.is_end
train_df, val_df, holdout_df = split.train_df, split.val_df, split.holdout_df
for label, part in (("train", train_df), ("val", val_df), ("holdout", holdout_df)):
    print(f"{label}: rows={len(part):,}  weeks={part['date'].nunique()}")

X_train = X.loc[train_df.index]
y_train = train_df[TARGET_COL]
X_val = X.loc[val_df.index]

search_rows = []
for alpha in ALPHA_GRID:
    model = Ridge(alpha=alpha, random_state=RANDOM_SEED)
    model.fit(X_train, y_train)
    pred_va = attach_scores(val_df, model.predict(X_val))
    sm = mean_date_ic(pred_va)
    search_rows.append(
        {
            "alpha": alpha,
            "val_mean_ic": sm["mean_ic"],
            "val_icir": sm["icir"],
            "n_dates": sm["n"],
        }
    )
    print(f"alpha={alpha:<6g}  val_ic={sm['mean_ic']:.4f}  ICIR={sm['icir']:.3f}")

search_df = (
    pd.DataFrame(search_rows)
    .sort_values(["val_mean_ic", "val_icir"], ascending=False)
    .reset_index(drop=True)
)
display(search_df)
BEST_ALPHA = float(search_df.iloc[0]["alpha"])
print(f"BEST_ALPHA={BEST_ALPHA}")

model = Ridge(alpha=BEST_ALPHA, random_state=RANDOM_SEED)
model.fit(X_train, y_train)
preds = attach_scores(df, model.predict(X))
if "feature_date" in df.columns:
    preds = preds.merge(
        df[["date", "ticker", "feature_date"]],
        on=["date", "ticker"],
        how="left",
    )
preds.to_parquet(PRED_PATH, index=False)
print(f"Saved predictions: {PRED_PATH}  rows={len(preds):,}")

ic_compare = ic_segment_table(preds, train_dates, val_dates, is_end)
display(ic_compare)


Masked week panel: rows=52,700 weeks=527 tickers=237 IS_weeks=344
After finite labels: 52,600 (from 52,700)
Complete-case drop: dropped=4,331  remain=48,269
train: rows=26,354  weeks=291
val: rows=4,814  weeks=52
holdout: rows=17,008  weeks=182
alpha=0.1     val_ic=-0.0141  ICIR=-0.046
alpha=1       val_ic=-0.0141  ICIR=-0.045
alpha=10      val_ic=-0.0140  ICIR=-0.045
alpha=100     val_ic=-0.0139  ICIR=-0.045


,alpha,val_mean_ic,val_icir,n_dates
0,100.0,-0.013874,-0.044614,52
1,10.0,-0.013978,-0.045042,52
2,1.0,-0.014109,-0.045481,52
3,0.1,-0.014132,-0.045561,52


BEST_ALPHA=100.0
Saved predictions: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\03_models\s1_equities\model_artifacts\s1_reshuffle_ridge_predictions.parquet  rows=48,269


,mean_ic,icir,n_dates
segment,,,
IS train,0.060715,0.276696,291
IS val,-0.013874,-0.044614,52
IS train+val,0.049407,0.208845,343
OS holdout,0.040293,0.142246,182
